In [ ]:
import requests

url = "https://api.sportradar.com/tennis/trial/v3/en/competitions.json"

headers = {
    "accept": "application/json",
    "x-api-key": "i2oSr6rhWNZa7hsdSZsDINmpVcj5sy1Xahg2sZuB"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    competitions_data = response.json()                    # Step - 1 : Convert the response into dictionary format.
    
    print(competitions_data)
else:
    print("Error in API:", response.status_code)

In [ ]:
competitions_data                   # Step - 2 : View the data and its structure.

In [3]:
# Step - 3 : PostgreSql connection and creating the table.

import psycopg2

connection = psycopg2.connect(
    host = "localhost",
    database = "sports",
    user = "postgres",
    password = "12345678",
    port = "5432" )

cursor = connection.cursor()

cursor.execute('''CREATE TABLE IF NOT EXISTS categories_table 
               (category_id VARCHAR(50)PRIMARY KEY,                 
               category_name VARCHAR(100) NOT NULL)''')


print("categories table created successfully")


cursor.execute(''' CREATE TABLE IF NOT EXISTS competitions_table (
                competition_id VARCHAR(50) PRIMARY KEY,
                competition_name VARCHAR(100) NOT NULL,
                parent_id VARCHAR(50),
                type VARCHAR(20) NOT NULL,
                gender VARCHAR(10) NOT NULL,
                category_id VARCHAR(50),
                FOREIGN KEY (category_id) REFERENCES categories_table(category_id)
)
''')

connection.commit()

print("competitions table created successfully")

categories table created successfully
competitions table created successfully


In [ ]:
# Step - 4 : Extract the competitions and categoires data from given data.

for comp in competitions_data['competitions']:

    competition_id = comp['id'].replace('sr:competition:','')

    category = comp.get('category')
       
    category_id = category['id'].replace('sr:category:','')

    category_name = category['name']

    gender = comp.get('gender', 'unknown',)

    parent_id = comp.get('parent_id')
    if parent_id is not None:
        parent_id = parent_id.replace('sr:competition:','')

# Step - 5 : Inserting the categories into the database.

    cursor.execute('''
        INSERT INTO categories_table (category_id, category_name)
        VALUES (%s, %s) ON CONFLICT (category_id) DO NOTHING         
    ''', (category_id, category_name))

# Step - 6 : Inserting the competitions into the database.

    cursor.execute('''
        INSERT INTO competitions_table 
        (competition_id, competition_name, parent_id, type, gender, category_id)
        VALUES (%s, %s, %s, %s, %s, %s) ON CONFLICT (competition_id) DO NOTHING
        ''', (competition_id,comp['name'],parent_id,comp['type'],gender,category_id))
    

connection.commit()

print("Categories and competitions data inserted successfully")

Categories and competitions data inserted successfully
